In [8]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_log_error as RMSE
from itertools import permutations


data_path = "train.csv"
data = pd.read_csv(data_path)
target_thing = np.log1p(data["hours_spent"])

input_data = data.drop(columns=[
    "project_id",
    "hours_spent",
    "log_hours",
    "hours_bucket"
])


qualitative_features = input_data.select_dtypes(include=["object", "string", "category"]).columns.tolist() 
#print(qualitative_features) => ['commit_cadence', 'primary_language', 'framework', 'has_custom_art', 'hosting_type', 'is_mobile_responsive', 'project_category', 'is_multiplayer', 'mentions_ai_api', 'is_clone_or_tutorial', 'author_prior_projects', 'author_account_age_years']

for col in qualitative_features: #fill in missing stuff just in case
    input_data[col] = input_data[col].fillna("Unknown").astype(str)

quantitative_features = input_data.select_dtypes(include=np.number).columns.tolist()

input_training, input_validation, output_training, output_validation = train_test_split(input_data, target_thing, test_size=0.2, random_state=42)
def evaluate(training, validation):
    magicModel = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.03,
        depth=6,
        loss_function="RMSE",
        verbose=100,
        early_stopping_rounds=100
    )

    magicModel.fit(
        training,
        output_training,
        cat_features=qualitative_features,
        eval_set=(validation, output_validation)
    )

    predictions = np.expm1(magicModel.predict(validation))
    predictions = np.maximum(predictions, 0) #no negatives

    true_hours = np.expm1(output_validation)

    return RMSE(true_hours,predictions)

baseline = evaluate(input_training, input_validation)
print("Baseline score:", baseline)

How_many_iterations = 250
learning_rate = .05
depth = 5
ratios = 15

print("screening")

results = []

ratio_count = (len(quantitative_features)* (len(quantitative_features) - 1))

current_ratio = 0


for a, b in permutations(quantitative_features, 2):

    current_ratio += 1

    name = f"{a}_div_{b}"

    print(
        f"\n[{current_ratio}/{ratio_count}] "
        f"Testing {name}"
    )

    train = input_training.copy()
    validation = input_validation.copy()

    train[name] = (
        train[a] / (train[b] + 1e-6)
    )

    validation[name] = (
        validation[a] / (validation[b] + 1e-6)
    )

    #stop infinite values
    train[name] = train[name].replace([np.inf, -np.inf],np.nan)

    validation[name] = validation[name].replace([np.inf, -np.inf],np.nan)

    validation = validation[train.columns]


    screening_model = CatBoostRegressor(
        iterations=How_many_iterations,
        learning_rate=learning_rate,
        depth=depth,
        loss_function="RMSE",

        # Don't print CatBoost's iteration-by-iteration output
        verbose=False,

        # Important: reproducible results
        random_seed=42,

        # Stop early if validation stops improving
        early_stopping_rounds=30
    )

    screening_model.fit(
        train,
        output_training,
        cat_features=qualitative_features,
        eval_set=(
            validation,
            output_validation
        )
    )

    predictions = np.expm1(screening_model.predict(validation))

    predictions = np.maximum(predictions,0)

    true_hours = np.expm1(output_validation)

    score = RMSE(true_hours,predictions)

    improvement = baseline - score

    results.append({
        "ratio": name,
        "screen_score": score,
        "screen_improvement": improvement
    })

    print(
        f"Score: {score:.6f} "
        f"({improvement:+.6f})"
    )

results = pd.DataFrame(results)

results = results.sort_values(
    "screen_score",
    ascending=True
)

print("best ratios")

print(results.head(20).to_string(index=False))

top_ratios = (
    results
    .head(ratios)
    ["ratio"]
    .tolist()
)

print(f"\nSelected top {ratios} ratios for full evaluation:")

for i, ratio in enumerate(top_ratios, 1):
    print(f"{i}. {ratio}")

print("\n" + "=" * 60)
print("FULL EVALUATION")
print("=" * 60)

final_results = []


for i, name in enumerate(top_ratios, 1):

    print(
        f"\n[{i}/{ratios}] "
        f"FULL TEST: {name}"
    )

    train = input_training.copy()
    validation = input_validation.copy()


    a, b = name.split("_div_")

    train[name] = (
        train[a] / (train[b] + 1e-6)
    )

    validation[name] = (
        validation[a] / (validation[b] + 1e-6)
    )

    train[name] = train[name].replace(
        [np.inf, -np.inf],
        np.nan
    )

    validation[name] = validation[name].replace(
        [np.inf, -np.inf],
        np.nan
    )
    validation = validation[train.columns]

    score = evaluate(
        train,
        validation
    )

    improvement = baseline - score

    final_results.append({
        "ratio": name,
        "score": score,
        "improvement": improvement
    })

    print(
        f"{name}: "
        f"{score:.6f} "
        f"({improvement:+.6f})"
    )

final_results = pd.DataFrame(final_results)

final_results = final_results.sort_values("score",ascending=True)

print("results")

print(final_results.to_string(index=False))


best = final_results.iloc[0]

print("\n" + "=" * 60)
print("BEST RATIO")
print("=" * 60)

print(f"Ratio:{best['ratio']}")

print(f"Score:       {best['score']:.6f}")

print(f"Baseline:    {baseline:.6f}")

print(f"Improvement: {best['improvement']:+.6f}")

Categorical features: 24
Numerical features: 45

Baseline score: 0.6586479208414179

PAIRWISE TRANSFORMATION SCREENING

[1/9900] ratio: num_commits, active_commit_days
Score: 0.660974 (-0.002326)

[2/9900] ratio: num_commits, project_span_days
Score: 0.661777 (-0.003129)

[3/9900] ratio: num_commits, pct_commits_weekend
Score: 0.664489 (-0.005841)

[4/9900] ratio: num_commits, pct_commits_late_night
Score: 0.662022 (-0.003374)

[5/9900] ratio: num_commits, lines_added_total
Score: 0.664997 (-0.006349)

[6/9900] ratio: num_commits, lines_deleted_total
Score: 0.665810 (-0.007162)

[7/9900] ratio: num_commits, churn_ratio
Score: 0.664320 (-0.005672)

[8/9900] ratio: num_commits, median_lines_per_commit
Score: 0.665983 (-0.007335)

[9/9900] ratio: num_commits, max_lines_in_one_commit
Score: 0.659973 (-0.001325)

[10/9900] ratio: num_commits, pct_code_in_first_commit
Score: 0.664979 (-0.006331)

[11/9900] ratio: num_commits, num_commit_authors
Score: 0.667984 (-0.009336)

[12/9900] ratio: n

KeyboardInterrupt: 